# Phần 3 — VLM & Prompting: chạy thử trên Kaggle

Notebook chạy `generate_json()` trên keyframe, sinh JSON metadata mô tả ảnh.

**Trước khi Run All, bật 2 thứ ở panel Settings bên phải:**
1. **Accelerator** → `GPU T4 x2` hoặc `GPU P100`
2. **Internet** → `On` (cần để tải model từ HuggingFace)

Thiếu GPU thì model chạy trên CPU — chậm gấp hàng chục lần.


## 1. Kiểm tra GPU

Luôn kiểm tra trước. Nếu ô này báo không có GPU, dừng lại và bật Accelerator.


In [ ]:
import torch

print('PyTorch :', torch.__version__)
print('Co GPU  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Ten GPU :', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'VRAM    : {vram:.1f} GB')
else:
    print('CANH BAO: chua bat GPU. Settings > Accelerator > GPU')


## 2. Cài thư viện

Kaggle có sẵn `torch` và `transformers`. Chỉ cần cài thêm phần lượng tử hóa 4-bit.
Mất khoảng 1-2 phút.


In [ ]:
!pip install -q "transformers>=4.51,<5" accelerate bitsandbytes qwen-vl-utils pydantic
import transformers
print('Cai xong | transformers:', transformers.__version__)


## 3. Nạp code Phần 3

Hai cách, ô dưới tự thử lần lượt:
- **Cách A**: clone repo nhóm (cần repo public hoặc đã cấu hình token)
- **Cách B**: upload thư mục `vlm_prompting` làm Kaggle Dataset rồi Add Data


In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/lolizabrett-byte/Multimodal-Agentic-Retrieval-Engine.git'
BRANCH   = 'research/vlm-prompting'
DICH     = Path('/kaggle/working/repo')

if not DICH.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, str(DICH)],
                   check=False)

PKG = DICH / 'system1' / 'research' / 'vlm_prompting'

if not PKG.exists():
    ung_vien = list(Path('/kaggle/input').glob('**/vlm_prompting'))
    if ung_vien:
        PKG = ung_vien[0]

assert PKG.exists(), f'Khong tim thay code tai {PKG}. Dung cach B: upload dataset.'
sys.path.insert(0, str(PKG))
print('Code tai:', PKG)


In [ ]:
from vlm import generate_json, thong_tin_moi_truong, goi_y_theo_vram, MODEL_REGISTRY

moi_truong = thong_tin_moi_truong()
print(moi_truong)

if moi_truong['vram_gb']:
    goi_y = goi_y_theo_vram(moi_truong['vram_gb'])
    print()
    print('Model chay duoc tren GPU nay:')
    for k in goi_y:
        s = MODEL_REGISTRY[k]
        print(f'  {k:<14} {s.vram_4bit_gb:>4.1f}GB  {s.ten_hien_thi}')


## 4. Chuẩn bị ảnh test

Ba nguồn, thử lần lượt:
1. Ảnh rời trong `/kaggle/input` (dataset đã Add Data)
2. File `.blob` — đọc thẳng 1 ảnh trong kho nén, **không giải nén cả kho** (tiết kiệm đĩa)
3. Ảnh mẫu tải từ internet — luôn chạy được


In [ ]:
import glob, zipfile
import io as _io
from PIL import Image

anh_test = None
nguon = None
DUOI_ANH = ('.jpg', '.jpeg', '.png', '.webp')

for duoi in ('jpg', 'jpeg', 'png', 'webp'):
    tim = glob.glob(f'/kaggle/input/**/*.{duoi}', recursive=True)
    if tim:
        anh_test = Image.open(tim[0]).convert('RGB')
        nguon = f'anh roi: {tim[0]}'
        break

if anh_test is None:
    blobs = glob.glob('/kaggle/input/**/*.blob', recursive=True)
    if blobs:
        zf = zipfile.ZipFile(blobs[0])
        ten = [n for n in zf.namelist() if n.lower().endswith(DUOI_ANH)]
        if ten:
            anh_test = Image.open(_io.BytesIO(zf.read(ten[0]))).convert('RGB')
            nguon = f'blob: {blobs[0]} -> {ten[0]}'

if anh_test is None:
    import urllib.request
    url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
    urllib.request.urlretrieve(url, '/kaggle/working/anh_mau.jpg')
    anh_test = Image.open('/kaggle/working/anh_mau.jpg').convert('RGB')
    nguon = 'anh mau COCO tai tu internet'

print('Nguon anh:', nguon)
print('Kich thuoc:', anh_test.size)
anh_test


## 5. Chạy thử MỘT ảnh

Bước quan trọng nhất. Một ảnh chạy vài giây; prompt sai thì sửa rồi chạy lại ngay.
**Đừng chạy 100 ảnh trước khi ô này ra kết quả sạch.**

Lần đầu sẽ tải model về (vài phút, model nặng 1-15GB tùy loại).


In [ ]:
import json, time
from vlm.generate import reset_vram_counter

MODEL = 'qwen25vl-3b'   # doi thanh 'vintern-1b' de thu model chuyen tieng Viet

reset_vram_counter()
bat_dau = time.time()

ket_qua = generate_json(anh_test, model_key=MODEL, debug=True)

print('=' * 60)
print(json.dumps(ket_qua, ensure_ascii=False, indent=2))
print('=' * 60)
print('Thoi gian:', ket_qua['_latency_sec'], 's')
print('VRAM dinh:', ket_qua.get('_vram_peak_gb'), 'GB')
print(f'Tong ke ca nap model: {time.time() - bat_dau:.1f}s')


## 6. Chế độ DEBUG — soi vài ảnh

Chạy 3-5 ảnh, in đầy đủ để kiểm tra bằng mắt trước khi chạy hàng loạt.

*Không phải cấu hình nào cũng phù hợp — phải dò trước khi đốt giờ GPU.*


In [ ]:
SO_ANH_DEBUG = 3

danh_sach = []
for duoi in ('jpg', 'jpeg', 'png', 'webp'):
    danh_sach += glob.glob(f'/kaggle/input/**/*.{duoi}', recursive=True)
danh_sach = danh_sach[:SO_ANH_DEBUG]

if not danh_sach:
    danh_sach = ['/kaggle/working/anh_mau.jpg']

for i, duong_dan in enumerate(danh_sach, 1):
    print(f'--- Anh {i}/{len(danh_sach)}: {duong_dan} ---')
    try:
        r = generate_json(duong_dan, model_key=MODEL)
        print('  doi tuong :', r['doi_tuong'])
        print('  mau sac   :', r['mau_sac'])
        print('  hanh dong :', r['hanh_dong'])
        print('  boi canh  :', r['boi_canh'])
        print('  caption   :', r['caption_chi_tiet'])
        print('  latency   :', r['_latency_sec'], 's')
    except Exception as e:
        print(f'  LOI: {type(e).__name__}: {e}')
    print()


## 7. So sánh các model (yêu cầu của đề bài)

Đề bài yêu cầu benchmark ít nhất 3 model. Ô này chạy cùng một ảnh qua nhiều model.

⚠️ Mỗi model tải 1-15GB. Chạy hết sẽ tốn thời gian và dung lượng đĩa Kaggle.


In [ ]:
DANH_SACH_MODEL = ['vintern-1b', 'qwen2vl-2b', 'qwen25vl-3b']

bang_ket_qua = []
for ten_model in DANH_SACH_MODEL:
    print(f'=== {ten_model} ===')
    reset_vram_counter()
    try:
        r = generate_json(anh_test, model_key=ten_model)
        bang_ket_qua.append({
            'model': ten_model,
            'latency_s': r['_latency_sec'],
            'vram_gb': r.get('_vram_peak_gb'),
            'json_hop_le': True,
            'caption': r['caption_chi_tiet'][:100],
        })
        print('  OK ', r['_latency_sec'], 's |', r['caption_chi_tiet'][:80])
    except Exception as e:
        bang_ket_qua.append({
            'model': ten_model,
            'json_hop_le': False,
            'loi': f'{type(e).__name__}: {str(e)[:100]}',
        })
        print(f'  LOI: {type(e).__name__}: {str(e)[:100]}')
    print()

import pandas as pd
pd.DataFrame(bang_ket_qua)


## 8. Lưu kết quả

File trong `/kaggle/working/` tải về được ở tab Output sau khi notebook chạy xong.

⚠️ **Phiên Kaggle tự ngắt sau 12 giờ.** Chạy hàng loạt phải lưu mỗi 25 ảnh,
không thì mất trắng cả phiên.


In [ ]:
from pathlib import Path

OUT = Path('/kaggle/working/vlm_smoke_results.json')
OUT.write_text(json.dumps({
    'moi_truong': moi_truong,
    'model_mac_dinh': MODEL,
    'ket_qua_mot_anh': ket_qua,
    'so_sanh_model': bang_ket_qua,
}, ensure_ascii=False, indent=2), encoding='utf-8')

print('Da luu:', OUT)
print('Tai ve o tab Output ben phai sau khi notebook chay xong.')


## 9. Huấn luyện QLoRA từ 500 caption chưng cất

Nạp `train.jsonl` (Phase 04, sinh bởi `scripts/dung_dataset_qlora.py`), gắn LoRA
vào Qwen2-VL-2B nén 4-bit, chỉ train tầng LLM (đóng băng vision encoder).

⚠️ **Kaggle mất sạch dữ liệu khi phiên tắt.** Sau khi train xong PHẢI bấm
**Save Version** và **tải file adapter zip về máy NGAY trong phiên** — đừng
hẹn "để lúc khác", phiên sau sẽ không còn gì.


In [ ]:
import torch

print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG CO')

# Bay da gap: chon GPU trong Settings giua phien KHONG co tac dung len phien
# dang chay. Neu assert duoi day fail -> Stop session, chon lai Accelerator
# GPU T4, roi Start session lai tu dau (khong Restart, phai Stop han).
assert torch.cuda.is_available(), "Chua bat GPU: Settings > Accelerator > GPU T4, roi Stop session va chay lai"


In [ ]:
# 'trl' keo theo transformers moi nhat, va ban 5.0 lam hong import.
# Nhac lai moc ghim o day de pip khong nang nguoc len sau cell 4.
!pip install -q -U peft bitsandbytes accelerate datasets trl "transformers>=4.51,<5"
import transformers
print('Cai xong | transformers:', transformers.__version__)


In [ ]:
from transformers import BitsAndBytesConfig, AutoProcessor

# Transformers ban moi doi ten AutoModelForVision2Seq -> AutoModelForImageTextToText.
# Giu try/except de notebook chay duoc ca tren moi truong da co san ban khac.
try:
    from transformers import AutoModelForImageTextToText as AutoModelVLM
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoModelVLM

QLORA_MODEL_HF_ID = 'Qwen/Qwen2-VL-2B-Instruct'  # khop vlm/model_registry.py key 'qwen2vl-2b'

# Cau hinh giong het vlm/model_loader.py:_tao_quant_config -- KHONG dung
# llm_int8_skip_modules: da do that tren T4, dung chung voi 4-bit gay loi
# "AssertionError: FP4 quantization state not initialized".
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# T4 KHONG co bf16 -- bat buoc float16 tuong minh o ca torch_dtype lan compute_dtype tren.
qlora_model = AutoModelVLM.from_pretrained(
    QLORA_MODEL_HF_ID,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map='auto',
)
qlora_processor = AutoProcessor.from_pretrained(QLORA_MODEL_HF_ID)
print('Da nap model + processor:', QLORA_MODEL_HF_ID)


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Dong bang toan bo model truoc, gan LoRA se tu bat lai grad cho cac tang duoc nham.
for tham_so in qlora_model.parameters():
    tham_so.requires_grad = False

qlora_model = prepare_model_for_kbit_training(qlora_model, use_gradient_checkpointing=True)

# Chi nham tang chieu (projection) cua phan LLM -- KHONG dam vao vision tower.
# Non-goal cua ke hoach cam train vision: phan 'mat' dang doc dung vat the,
# cai sai la phan 'mieng' viet cau, dung dam vao mat.
LORA_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=LORA_TARGET_MODULES,
)

qlora_model = get_peft_model(qlora_model, lora_config)
qlora_model.print_trainable_parameters()

# Bang chung so cho viec vision da bi dong bang that: ty le tham so train duoc
# phai duoi 1% tong so. Con so lon nghia la LoRA lo nham vao vision tower.
_tong = sum(p.numel() for p in qlora_model.parameters())
_train_duoc = sum(p.numel() for p in qlora_model.parameters() if p.requires_grad)
_ty_le = _train_duoc / _tong
print(f'Ty le tham so train duoc: {_ty_le:.4%}')
assert _ty_le < 0.01, f'Vision encoder chua dong bang that: {_ty_le:.4%} >= 1%'


In [ ]:
import json
from pathlib import Path
from PIL import Image

# Kaggle Dataset gan qua Add Data -- duong dan thuc te tuy ten dataset da upload,
# sua DATASET_DIR neu khac. Dataset phai o che do Private (chua keyframe cuoc thi).
DATASET_DIR = Path('/kaggle/input/aic-vlm-distill-290')
TRAIN_JSONL = DATASET_DIR / 'train.jsonl'
IMAGES_DIR = DATASET_DIR / 'images'

def doc_jsonl(duong_dan):
    return [json.loads(dong) for dong in duong_dan.read_text(encoding='utf-8').splitlines()]

mau_train = doc_jsonl(TRAIN_JSONL)
print('So mau train:', len(mau_train))


def collate_qlora(batch):
    """Ghep anh + hoi thoai qua processor cua Qwen2-VL thanh 1 batch tensor."""
    anh_list = [Image.open(IMAGES_DIR / m['image']).convert('RGB') for m in batch]
    text_list = [
        qlora_processor.apply_chat_template(m['messages'], tokenize=False, add_generation_prompt=False)
        for m in batch
    ]
    enc = qlora_processor(text=text_list, images=anh_list, return_tensors='pt', padding=True)
    enc['labels'] = enc['input_ids'].clone()
    return enc


In [ ]:
from transformers import TrainingArguments, Trainer
from torch.utils.data import Dataset as TorchDataset


class QloraDataset(TorchDataset):
    def __init__(self, mau_list):
        self.mau_list = mau_list

    def __len__(self):
        return len(self.mau_list)

    def __getitem__(self, idx):
        return self.mau_list[idx]


training_args = TrainingArguments(
    output_dir='/kaggle/working/lora-qwen2vl-2b-haiku500',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,   # bu batch nho -- 14.6GB khong cho batch lon voi anh
    num_train_epochs=2,              # >3 epoch tren 450 mau de hoc thuoc long thay vi hoc phong cach
    learning_rate=1e-4,
    fp16=True,                       # T4 khong co bf16
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    save_strategy='epoch',
    logging_steps=10,
    report_to='none',
)

trainer = Trainer(
    model=qlora_model,
    args=training_args,
    train_dataset=QloraDataset(mau_train),
    data_collator=collate_qlora,
)

ket_qua_train = trainer.train()
print(ket_qua_train)
# Loss phai giam dan qua cac logging_steps o tren. Loss dung im hoac thanh nan
# -> dung ngay, kiem lai dtype (phai float16, khong bf16) va learning_rate.


In [ ]:
import shutil

ADAPTER_DIR = Path('/kaggle/working/lora-qwen2vl-2b-haiku500')
qlora_model.save_pretrained(ADAPTER_DIR)
qlora_processor.save_pretrained(ADAPTER_DIR)

ZIP_PATH = shutil.make_archive('/kaggle/working/lora-qwen2vl-2b-haiku500', 'zip', ADAPTER_DIR)
print('Da nen adapter:', ZIP_PATH)

# BAM 'Save Version' NGAY BAY GIO roi tai file zip o tab Output ve may.
# Kaggle mat sach /kaggle/working khi phien tat -- doi sang phien sau la mat trang.


## Checklist sau khi train

1. Tải file `lora-qwen2vl-2b-haiku500.zip` về máy (tab Output, sau khi Save Version).
2. Giải nén, đặt nội dung vào `results/lora-qwen2vl-2b-haiku500/` ở repo local.
3. Kiểm thư mục có đủ `adapter_config.json` + `adapter_model.safetensors`.
4. Chạy Phase 05 để đo trước/sau (nạp adapter vào `vlm/adapters.py`, so sánh với
   tập giữ lại `results/danh_sach_anh_holdout.txt`).
